## Forecast Reader

In [1]:
import sys
sys.path.append('..')  # Add project root
from src.forecast_reader import ForecastReader

In [2]:
forecast_path = "../data/ibm_forecast.xlsx"
forecast_reader = ForecastReader(forecast_path)

forecast_reader.load_forecast()

result = forecast_reader.get_forecast_data()
result

{'9500614556': {'Jan': {'Forecast': 0.0, 'Source': []},
  'Feb': {'Forecast': 0.0, 'Source': []},
  'Mar': {'Forecast': 0.0, 'Source': []},
  'Apr': {'Forecast': 0.0, 'Source': []},
  'May': {'Forecast': 0.0, 'Source': []},
  'Jun': {'Forecast': 0.0, 'Source': []},
  'Jul': {'Forecast': 0.0, 'Source': []},
  'Aug': {'Forecast': 0.0, 'Source': []},
  'Sep': {'Forecast': 0.0, 'Source': []},
  'Oct': {'Forecast': 0.0, 'Source': []},
  'Nov': {'Forecast': 0.0, 'Source': []},
  'Dec': {'Forecast': 0.0, 'Source': []}},
 '9500705161': {'Jan': {'Forecast': 0.0, 'Source': []},
  'Feb': {'Forecast': 0.0, 'Source': []},
  'Mar': {'Forecast': 0.0, 'Source': []},
  'Apr': {'Forecast': 0.0, 'Source': []},
  'May': {'Forecast': 0.0, 'Source': []},
  'Jun': {'Forecast': 0.0, 'Source': []},
  'Jul': {'Forecast': 0.0, 'Source': []},
  'Aug': {'Forecast': 0.0, 'Source': []},
  'Sep': {'Forecast': 0.0, 'Source': []},
  'Oct': {'Forecast': 0.0, 'Source': []},
  'Nov': {'Forecast': 0.0, 'Source': []},
  'De

## Template Writer

In [6]:
import sys
sys.path.append('..')  # Add project root
from src.template_writer import TemplateWriter

In [7]:
tw = TemplateWriter("../data/Financial Spreadsheet Template v2.xlsx")
tw.parse_headers()  # Build header map
tw.write_forecast(result, po_filter=['9500879389', '9500882917'])  # Write forecast data
tw.save("../data/completed_template.xlsx")  # Save updated template

## Integration Testing 

In [1]:
import sys
sys.path.append('..')  # Add project root
from src.forecast_reader import ForecastReader
from src.transactional_detail_reader import InvoiceActualReader, AccrualReader
from src.template_writer import FinancialTemplateV2Writer


In [2]:
forecast_path = "../data/ibm_forecast.xlsx"
tdf_path = "../data/C-TIES AP09 2025.xlsx"
template_path = "../data/Financial Spreadsheet Template v2.xlsx"

forecast_reader = ForecastReader(forecast_path)
forecast_reader.load_forecast()
forecast_data = forecast_reader.get_forecast_data()
print("Loaded forecast data\n")

actual_reader = InvoiceActualReader(tdf_path)
actual_reader.load_transactional_detail()
actual_data = actual_reader.get_transactional_data()
print("Loaded actuals data\n")


accrual_reader = AccrualReader(tdf_path)
accrual_reader.load_transactional_detail()
accrual_data = accrual_reader.get_transactional_data()
print("Loaded accrual data\n")

Loaded forecast data

Loaded actuals data

Loaded accrual data



In [3]:
# Function to combine forecasts, actuals, and accrual data into one dictionary
def combine_data(forecast, actual, accrual):
    combined = {}
    all_pos = set(forecast.keys()) | set(actual.keys()) | set(accrual.keys())

    for po in all_pos:
        combined[po] = {}
        months = set()
        for d in [forecast, actual, accrual]:
            if po in d:
                months.update(d[po].keys())

        for month in months:
            combined[po][month] = {
                'Forecast': forecast.get(po, {}).get(month, {}).get('Forecast', 0),
                'Actual': actual.get(po, {}).get(month, {}).get('Actual', 0),
                'Accrual': accrual.get(po, {}).get(month, {}).get('Accrual', 0),
                'Accrual Reversal': accrual.get(po, {}).get(month, {}).get('Accrual Reversal', 0),
                '2WM': accrual.get(po, {}).get(month, {}).get('2WM', False),
                # Sources can be combined or kept separate if needed
                # TODO: Add source field and decide how to handle sources
            }

    # Reordering months for easier debugging
    month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    for po in combined:
        # Sort the inner dict by month_order
        combined[po] = {month: combined[po][month] for month in month_order if month in combined[po]}
    return combined

In [6]:
result = combine_data(forecast_data, actual_data, accrual_data)

# List of POs you want to write
selected_pos = ['9500879389', '9500882917']

# Filter the result dictionary
filtered_result = {po: data for po, data in result.items() if po in selected_pos}


In [10]:
template_writer = FinancialTemplateV2Writer(template_path)

template_writer.parse_template()

template_writer.write_data(filtered_result)

template_writer.save("template_output.xlsx")